## Mujoco Jacobian
- Get Jacobian of the specific body
- Get pseudo-inverse of Jacobian
    - Damped Least Squares
    - Singular Value Decomposition

### Singular Value Decomposition
- Decompose jacobian into S, V and sigma
- V: direction of motion in joint space, U: corresponding motion direction in end-effector space
- Sigma: Contains singular values (degree of amplification)
    - large singular value -> small joint move makes large 
    - Thresholding sigma to avoid singular value
- $J = U \Sigma V^T$
- $J^+ = V \Sigma^+ U^T$
</br>

### Damped Least Squares
- pseudo-inverse of Jacobian
- Damping factor: reduce singularity issue
    - lambda damping
    - e with error vector
- $\Delta\theta = J^T(JJ^T + \lambda^2I)^{-1} \vec{e}$

Create UR environment

In [ ]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [2]:
model_path = "../ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

In [6]:
def get_body_name (model, data):
    body_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_BODY, body_idx) for body_idx in range(model.nbody)]
    return body_names

body_names = get_body_name(model, data)

print(body_names)

['world', 'base', 'shoulder_link', 'upper_arm_link', 'forearm_link', 'wrist_1_link', 'wrist_2_link', 'wrist_3_link']


Get Mujoco Jacobian

In [7]:
""" GET MUJOCO JACOBIAN """

body_name = "wrist_3_link"

# initialize positional & rotational jacobian
Jacobian_p = np.zeros((3,model.nu))
Jacobian_r = np.zeros((3,model.nu))
# get jacobian of end-effector
mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
print(Jacobian_p)
# mujoco.mj_jac(model, data, p, r, )
# mj jacbody inherits mj_jac
# mj jac can specify the point

[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]


In [9]:
""" GET MUJOCO JACOBIAN """

data.ctrl = [1.0] * 6
for i in range (10):
    mujoco.mj_step(model, data)

mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
print(Jacobian_p)
# mujoco.mj_jac(model, data, p, r, )
# mj jacbody inherits mj_jac
# mj jac can specify the point

[[-8.12553339e-01  6.73912670e-04  6.63647624e-04  6.13081186e-04
   0.00000000e+00  0.00000000e+00]
 [-1.38987315e-01 -1.09852094e-01 -1.08178825e-01 -9.99361711e-02
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -8.13390683e-01 -3.88393977e-01  3.51935061e-03
  -4.33680869e-19  0.00000000e+00]]


### Main loop

In [ ]:
"""
function: name and value array to input control  
"""

def apply_control_name (model, data, name, value):

    if len(name) != len(value):
        raise ValueError("length of name and value is different")

    control_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_ACTUATOR,ctrl_idx) for ctrl_idx in range(model.nu)]
    ctrl_ = np.zeros(model.nu)

    for i, n in enumerate(name):
        if n in control_names:
            idx = control_names.index(n)
            ctrl_[idx] = value[i]
        else:
            print(f"Name {n} is not included in actuator names, passing..")

    data.ctrl = ctrl_
    return None

array([0. , 0.8, 0.8, 0. , 0. , 0. ])

In [ ]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)
mujoco.mj_resetData(model, data)

while True:
    if viewer.is_alive:

        # apply control
        apply_control_name(model, data, name=["shoulder_lift", "elbow"], value=[0.8, 0.8]) # now data 
        # control_signal = [0.] * 6
        
        mujoco.mj_step(model, data)
        viewer.render()

    else:
        break

# close
viewer.close()

Pressed ESC
Quitting.
